# 1.1.10 Object-Oriented Programming

Build an ML estimator class hierarchy — abstract base, KNN, Naive Bayes, Pipeline.

In [ ]:
import urllib.request, csv, math
from pathlib import Path
from collections import Counter

DATA = Path('data'); DATA.mkdir(exist_ok=True)
dest = DATA/'iris.csv'
if not dest.exists():
    urllib.request.urlretrieve('https://huggingface.co/datasets/scikit-learn/iris/resolve/main/Iris.csv', dest)

with open(dest, newline='') as f: rows = list(csv.DictReader(f))
X = [[float(r['SepalLengthCm']),float(r['SepalWidthCm']),float(r['PetalLengthCm']),float(r['PetalWidthCm'])] for r in rows]
y = [r['Species'] for r in rows]
print(f'Iris: {len(X)} samples, classes={set(y)}')

In [ ]:
from abc import ABC, abstractmethod
class BaseEstimator(ABC):
    _fitted = False
    @abstractmethod
    def fit(self, X, y): ...
    @abstractmethod
    def predict(self, X): ...
    def score(self, X, y):
        return sum(a==b for a,b in zip(self.predict(X),y))/len(y)
    def __call__(self, X): return self.predict(X)

class KNN(BaseEstimator):
    def __init__(self, k=5):
        self.k = k
    def __repr__(self): return f'KNN(k={self.k})'
    def fit(self, X, y):
        self._X, self._y = X, y; self._fitted = True; return self
    def predict(self, X):
        def dist(a,b): return math.sqrt(sum((ai-bi)**2 for ai,bi in zip(a,b)))
        return [Counter(l for _,l in sorted([(dist(x,xt),yt) for xt,yt in zip(self._X,self._y)])[:self.k]).most_common(1)[0][0] for x in X]

print(KNN(k=3))

In [ ]:
import random; random.seed(42)
idx = list(range(len(X))); random.shuffle(idx)
cut = int(len(X)*.8)
Xtr,Xte,ytr,yte = [X[i] for i in idx[:cut]],[X[i] for i in idx[cut:]],[y[i] for i in idx[:cut]],[y[i] for i in idx[cut:]]

for k in [1, 3, 5, 7]:
    model = KNN(k=k).fit(Xtr, ytr)
    print(f'KNN(k={k})  acc={model.score(Xte, yte):.4f}')

In [ ]:
# __call__ dunder
knn = KNN(k=5).fit(Xtr, ytr)
preds = knn(Xte[:5])  # calls predict via __call__
print('Predicted:', preds)
print('Actual   :', yte[:5])

In [ ]:
# Inheritance chain
print('MRO:', [c.__name__ for c in KNN.__mro__])